<a href="https://colab.research.google.com/github/zeinafarghaly-arch/ML-flyrank/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [5]:
from google.colab import files

uploaded = files.upload()

Saving content_refresh_anonymized.csv to content_refresh_anonymized.csv


In [6]:
import pandas as pd

df = pd.read_csv("/content/content_refresh_anonymized.csv")

In [7]:
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [8]:
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## My Rule

I will prioritize content that has not been updated for a long time and also has high search volume.

Reason codes:
- REFRESH_STALE: Content has not been updated for a long time.
- QUICK_WIN: Content has high search volume.

In [9]:
import pandas as pd

stale_table = (
    df.groupby("freshness_tier")
      .agg(
          n=("content_id", "count"),
          avg_trend=("trend_pct", "mean")
      )
)

print("Signal 1: Freshness")
print(stale_table)

volume_table = (
    df.groupby("impression_tier")
      .agg(
          n=("content_id", "count"),
          avg_search_volume=("search_volume", "mean")
      )
)

print("\nSignal 2: Search Volume")
print(volume_table)

Signal 1: Freshness
                    n  avg_trend
freshness_tier                  
0-30            20480   0.784405
181+              174  -6.781203
31-90             175  -7.373054
91-180           9171 -15.683224

Signal 2: Search Volume
                     n  avg_search_volume
impression_tier                          
excellent         1078         115.023343
good              7205         163.134119
low              11248         138.062185
moderate         10469         179.162663


Verdict 1: CONFIRMED
Older content generally shows higher opportunity, so I included staleness.

Verdict 2: CONFIRMED
Higher search volume indicates greater potential impact, so I included it.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [10]:
df["stale_score"] = df["days_since_last_update"] / df["days_since_last_update"].max()
df["volume_score"] = df["search_volume"] / df["search_volume"].max()

df["baseline_score"] = (
    0.6 * df["stale_score"] +
    0.4 * df["volume_score"]
)

df["reason_code"] = "REFRESH_STALE"

df.loc[
    df["search_volume"] >
    df["search_volume"].median(),
    "reason_code"
] = "QUICK_WIN"

df["action"] = "Monitor"

df.loc[
    df["baseline_score"] >= 0.60,
    "action"
] = "Refresh"

queue = df.sort_values(
    "baseline_score",
    ascending=False
)

queue.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,stale_score,volume_score,baseline_score,reason_code,action
26242,content_55a5b1c46474,client_4ec9599fc2,0.0,0.0,LOW,0.0,keyword article,informational,NaN,NaN,...,0.0,low,page_1,down,-88.5,1.000000,0.0,0.600000,REFRESH_STALE,Refresh
4606,content_3f3576c295f5,client_4ec9599fc2,0.0,0.0,LOW,0.0,keyword article,informational,NaN,NaN,...,0.0,low,top_3,flat,NaN,1.000000,0.0,0.600000,REFRESH_STALE,Refresh
29384,content_f6fdf87348f6,client_4ec9599fc2,0.0,0.0,LOW,0.0,keyword article,informational,NaN,NaN,...,0.0,low,page_3_5,down,-100.0,1.000000,0.0,0.600000,REFRESH_STALE,Refresh
18440,content_8d56efff1e71,client_4ec9599fc2,0.0,0.0,LOW,0.0,keyword article,informational,NaN,NaN,...,0.0,low,page_3_5,new,NaN,0.997319,0.0,0.598391,REFRESH_STALE,Monitor
24216,content_1b4ec72dafd4,client_4ec9599fc2,0.0,0.0,LOW,0.0,keyword article,informational,NaN,NaN,...,0.0,low,page_1,down,-100.0,0.997319,0.0,0.598391,REFRESH_STALE,Monitor


In [11]:
import os

os.makedirs("/content/work/outputs", exist_ok=True)

queue.to_csv(
    "/content/work/outputs/baseline_action_score.csv",
    index=False
)

print("CSV saved successfully!")

CSV saved successfully!


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [12]:
top20 = queue.head(20)

top20[
    [
        "content_id",
        "baseline_score",
        "action",
        "reason_code",
        "days_since_last_update",
        "search_volume"
    ]
]

,content_id,baseline_score,action,reason_code,days_since_last_update,search_volume
26242,content_55a5b1c46474,0.600000,Refresh,REFRESH_STALE,373,0.0
4606,content_3f3576c295f5,0.600000,Refresh,REFRESH_STALE,373,0.0
29384,content_f6fdf87348f6,0.600000,Refresh,REFRESH_STALE,373,0.0
18440,content_8d56efff1e71,0.598391,Monitor,REFRESH_STALE,372,0.0
24216,content_1b4ec72dafd4,0.598391,Monitor,REFRESH_STALE,372,0.0
12140,content_ef99c4abd9ab,0.567292,Monitor,QUICK_WIN,104,74000.0
21984,content_02b0d6e30129,0.504080,Monitor,QUICK_WIN,313,110.0
7509,content_7a888d3d99c8,0.503972,Monitor,QUICK_WIN,313,90.0
15790,content_6476d1d8c050,0.503539,Monitor,REFRESH_STALE,313,10.0
18841,content_94991fe6268c,0.503539,Monitor,REFRESH_STALE,313,10.0


| Rank | Action | Reason Code | Confidence Note | What would make it wrong? |
|------|--------|-------------|-----------------|---------------------------|
| 1 | Refresh | REFRESH_STALE | High confidence because the content has not been updated for 373 days. | The page may still be accurate and not require an update. |
| 2 | Refresh | REFRESH_STALE | High confidence because the content is very stale. | The update date could be incorrect or missing. |
| 3 | Refresh | REFRESH_STALE | High confidence due to long time since last update. | The content may be evergreen and still relevant. |
| 4 | Monitor | REFRESH_STALE | Medium confidence because the page is almost as stale but scored slightly lower. | A recent unpublished update may not be reflected in the data. |
| 5 | Monitor | REFRESH_STALE | Medium confidence for similar reasons as Rank 4. | The content quality may already be sufficient. |
| 6 | Monitor | QUICK_WIN | High search volume suggests a potentially valuable opportunity. | High search volume alone does not guarantee that refreshing the page will improve performance. |
| 7 | Monitor | QUICK_WIN | Moderate confidence because the page is old and has some search demand. | Search demand may be seasonal or temporary. |
| 8 | Monitor | QUICK_WIN | Moderate confidence due to reasonable staleness and search volume. | User intent may have changed since the data was collected. |
| 9 | Monitor | REFRESH_STALE | Moderate confidence because the page is stale. | The page could already rank well and not benefit from changes. |
| 10 | Monitor | REFRESH_STALE | Moderate confidence because it has not been updated recently. | The page may already satisfy users' needs. |
| 11 | Monitor | QUICK_WIN | Good opportunity because of high search volume. | The page might already have a high CTR and not need intervention. |
| 12 | Monitor | QUICK_WIN | Similar reasoning to Rank 11. | Search volume may not translate into meaningful business value. |
| 13 | Monitor | QUICK_WIN | Moderate confidence based on demand. | The keyword competition could limit gains. |
| 14 | Monitor | QUICK_WIN | Moderate confidence because of search opportunity. | Performance improvements may be minimal despite high volume. |
| 15 | Monitor | QUICK_WIN | Moderate confidence because the rule identified search demand. | Search volume may fluctuate over time. |
| 16 | Monitor | REFRESH_STALE | Moderate confidence because the page is over 300 days old. | The page may already contain evergreen information. |
| 17 | Monitor | REFRESH_STALE | Moderate confidence because of staleness. | Metadata could be inaccurate. |
| 18 | Monitor | REFRESH_STALE | Moderate confidence because the page has not been updated for a long time. | Another factor not included in the rule may be more important. |
| 19 | Monitor | REFRESH_STALE | Moderate confidence because the page appears stale. | Business priorities may differ from the rule. |
| 20 | Monitor | REFRESH_STALE | Moderate confidence due to long time since last update. | The rule only considers two signals and may miss other important factors. |

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak Picks + Leakage Check

### Weak Picks

Some of the highest-ranked pages have **0 search volume** but appear at the top because they have not been updated for a long time. This suggests that my rule places too much weight on content staleness.

Other pages with very high search volume received lower scores because they were updated more recently. In practice, these pages could still be good refresh candidates.

In future iterations, I would improve the rule by including additional signals such as **trend_pct**, **impressions_90d**, or **ctr** so that the ranking better balances freshness and potential impact.

### Leakage Check

I confirmed that my baseline score only uses current features:

- `days_since_last_update`
- `search_volume`

I did **not** use:
- future performance windows,
- product-generated flags,
- labels,
- or any information that would leak the desired outcome into the score.

The rule is based only on information that would have been available at the time of making the decision.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.